# Elo rating baseline

This notebook builds MatchCast's chronological Elo rating system on top of
`data/processed/matches.csv` (see `02_data_cleaning.ipynb`). It is built up
section by section, one concern at a time, starting with the rating
specification itself.

## Elo specification

**Initial rating.** Every team starts at `INITIAL_RATING = 1500` the first
time it appears, regardless of era. This is the standard neutral Elo
starting point; teams converge to their true strength as matches accumulate.

**K-factor.** `K_FACTOR = 20` for every match. A fixed, moderate K keeps
ratings responsive to results without an extra tuned parameter for match
importance in this baseline; a tournament-weighted K is a documented future
extension, not part of this baseline.

**Home advantage.** `HOME_ADVANTAGE = 100` rating points are added to the
home team's rating only for computing the *expected* score of the match. The
underlying stored ratings are not shifted; the adjustment is applied purely
inside the expected-score formula below.

**Neutral-venue behavior.** When a match's `neutral` flag is `True`,
`HOME_ADVANTAGE` is not applied (treated as `0`), because neither side is
playing at home.

**Expected score.** For a match with (adjusted) home rating `R_h` and away
rating `R_a`:

```text
E_home = 1 / (1 + 10 ** (-(R_h - R_a) / 400))
E_away = 1 - E_home
```

**Actual score.** `S = 1` for a win, `0.5` for a draw, `0` for a loss, from
each team's own perspective.

**Update formula.** After the match:

```text
new_R_home = R_home + K_FACTOR * (S_home - E_home)
new_R_away = R_away + K_FACTOR * (S_away - E_away)
```

Ratings are updated only after that match's pre-match features have been
recorded (implemented in a later section), so no match ever uses information
from its own outcome or from a future match.

In [1]:
INITIAL_RATING = 1500.0
K_FACTOR = 20.0
HOME_ADVANTAGE = 100.0


def expected_score(rating_a: float, rating_b: float) -> float:
    """Probability that the side with rating_a beats the side with rating_b."""
    return 1.0 / (1.0 + 10 ** (-(rating_a - rating_b) / 400.0))


def match_expected_scores(
    home_rating: float, away_rating: float, neutral: bool
) -> tuple[float, float]:
    """Return (expected_home_score, expected_away_score) for a match."""
    advantage = 0.0 if neutral else HOME_ADVANTAGE
    expected_home = expected_score(home_rating + advantage, away_rating)
    return expected_home, 1.0 - expected_home


def updated_ratings(
    home_rating: float,
    away_rating: float,
    home_actual_score: float,
    neutral: bool,
) -> tuple[float, float]:
    """Return (new_home_rating, new_away_rating) after a completed match."""
    expected_home, expected_away = match_expected_scores(home_rating, away_rating, neutral)
    away_actual_score = 1.0 - home_actual_score
    new_home = home_rating + K_FACTOR * (home_actual_score - expected_home)
    new_away = away_rating + K_FACTOR * (away_actual_score - expected_away)
    return new_home, new_away

### Validation: Elo specification

In [2]:
# Expected scores are complementary probabilities.
for neutral in (True, False):
    e_home, e_away = match_expected_scores(1600.0, 1500.0, neutral)
    assert abs((e_home + e_away) - 1.0) < 1e-12
    assert 0.0 < e_home < 1.0 and 0.0 < e_away < 1.0

# Equal ratings on a neutral pitch imply a 50/50 expectation.
e_home, e_away = match_expected_scores(1500.0, 1500.0, neutral=True)
assert abs(e_home - 0.5) < 1e-12
assert abs(e_away - 0.5) < 1e-12

# Home advantage favors the home side when ratings are otherwise equal.
e_home_adv, _ = match_expected_scores(1500.0, 1500.0, neutral=False)
assert e_home_adv > 0.5

# A won match increases the winner's rating and decreases the loser's.
new_home, new_away = updated_ratings(1500.0, 1500.0, home_actual_score=1.0, neutral=True)
assert new_home > 1500.0
assert new_away < 1500.0
# Rating points are conserved in a symmetric (neutral, equal-rating) update.
assert abs((new_home - 1500.0) - (1500.0 - new_away)) < 1e-9

# A draw between equally rated teams on a neutral pitch leaves ratings unchanged.
draw_home, draw_away = updated_ratings(1500.0, 1500.0, home_actual_score=0.5, neutral=True)
assert abs(draw_home - 1500.0) < 1e-9
assert abs(draw_away - 1500.0) < 1e-9

assert INITIAL_RATING > 0
assert K_FACTOR > 0
assert HOME_ADVANTAGE >= 0

print("Elo specification validated.")

Elo specification validated.
